#1. Загрузка и подготовка типов

In [ ]:
import numpy as np

In [ ]:
dtypes = [('col1', 'i8'), ('col2', 'i4'), ('col3', 'f8'),
          ('col4', 'f8'), ('col5', 'f8'), ('col6', 'i4')]

In [ ]:


df = np.genfromtxt("/data.csv", names=True, dtype=dtypes, delimiter=",",
                   missing_values='', filling_values=np.nan,
                   encoding='utf-8', invalid_raise=False)



In [ ]:
df

array([(1600000180, 35, 75.74629  ,  12.966491 ,  7.4050336, 15),
       (1600000255, 71,  3.2269726, -14.408976 ,  3.6415694, 53),
       (1600000261, 86, 43.881695 ,  -5.9387364,  0.3374592, 86), ...,
       (1749999870, 55, 27.15272  ,  -8.469389 ,  7.615275 , 50),
       (1749999928, 81, 67.86183  ,  19.257215 , -7.705005 , 53),
       (1749999937, 26, 53.15101  ,  -2.4991076,  4.074159 , 58)],
      dtype=[('ts', '<i8'), ('athlete_id', '<i4'), ('dist', '<f8'), ('pace', '<f8'), ('cal', '<f8'), ('zone', '<i4')])

In [ ]:
df[10:-1]

array([(1600000820, 15, 84.687454 , 11.126777 , -1.8943539 ,  6),
       (1600000860, 75, 52.2305   ,  1.4934486,  9.048953  , 72),
       (1600001101, 52, 51.462936 ,  7.4628778, -8.920704  , 39), ...,
       (1643463656, 83,        nan, -6.6608458,  9.717797  , 69),
       (1643463727, 30,  1.9334767, -2.4051986, -0.85869604, 88),
       (1643463748, 73, 90.88975  ,  3.3130171, -5.310952  , 34)],
      dtype=[('ts', '<i8'), ('athlete_id', '<i4'), ('dist', '<f8'), ('pace', '<f8'), ('cal', '<f8'), ('zone', '<i4')])

In [ ]:
df.dtype

dtype([('ts', '<i8'), ('athlete_id', '<i4'), ('dist', '<f8'), ('pace', '<f8'), ('cal', '<f8'), ('zone', '<i4')])

In [ ]:
df["dist"].dtype
print(type(df["dist"].dtype))

<class 'numpy.dtypes.Float64DType'>


In [ ]:
df.shape

(2000000,)

Объём занимаемой памяти в байтах и Мб

In [ ]:
df.nbytes

80000000

In [ ]:
df.nbytes / 1024

78125.0

In [ ]:
columns = ["ts", "athlete_id", "dist", "pace", "cal", "zone"]

for col in columns:
  mask = np.isnan(df[col]) | np.isinf(df[col])
  percent = (mask.sum() / len(df[col])) * 100

  print(f"В столбце {col} процент{percent} {mask.sum()} NaN и Inf.", end = " ")
  if percent > 3:
    print(f"В столбце содержится больше 3%!")
  else:
    print()



В столбце ts процент0.0 0 NaN и Inf. 
В столбце athlete_id процент0.0 0 NaN и Inf. 
В столбце dist процент3.0088 60176 NaN и Inf. В столбце содержится больше 3%!
В столбце pace процент2.99315 59863 NaN и Inf. 
В столбце cal процент2.9770000000000003 59540 NaN и Inf. 
В столбце zone процент0.0 0 NaN и Inf. 


#2. Векторизованная фильтрация и очистка

Доля от общего набора

In [ ]:
anomaly_mask = (df['dist'] < 0) | (df['pace'] < 0) | (df['cal'] < 0)
print(anomaly_mask)
n_anomalies = anomaly_mask.sum()
frac_anomalies = n_anomalies / len(df)

print(frac_anomalies)

[False  True  True ...  True  True  True]
0.7416195


Очистка данных

In [ ]:
mask = (df['dist'] < 0) | (df['pace'] < 0)
df_clean = df[~mask].copy()


df_clean['cal'] = np.where(df_clean['cal'] < 0, 0, df_clean['cal'])


q10, q90 = np.nanpercentile(df_clean['pace'], [10, 90])
df_clean['pace'] = np.clip(df_clean['pace'], q10, q90)

In [ ]:
df_clean["pace"]

array([12.966491 ,  4.47056  ,  1.3435365, ...,  1.6444373,  7.3264446,
       19.257215 ])

#3. Группировка + Нормализация

In [ ]:
athlete_group = np.unique(df_clean["athlete_id"])

print(f"Кол-во групп {len(athlete_group)}")

for group in athlete_group:
  print(f"Мощность группы {group} = {len(df_clean[df_clean["athlete_id"] == group])}")

Кол-во групп 100
Мощность группы 0 = 10050
Мощность группы 1 = 10003
Мощность группы 2 = 10030
Мощность группы 3 = 10167
Мощность группы 4 = 10012
Мощность группы 5 = 9853
Мощность группы 6 = 10142
Мощность группы 7 = 10108
Мощность группы 8 = 10121
Мощность группы 9 = 10010
Мощность группы 10 = 10033
Мощность группы 11 = 9869
Мощность группы 12 = 9927
Мощность группы 13 = 10052
Мощность группы 14 = 10028
Мощность группы 15 = 10150
Мощность группы 16 = 10190
Мощность группы 17 = 10087
Мощность группы 18 = 10025
Мощность группы 19 = 9891
Мощность группы 20 = 10046
Мощность группы 21 = 9952
Мощность группы 22 = 10161
Мощность группы 23 = 9924
Мощность группы 24 = 9942
Мощность группы 25 = 9925
Мощность группы 26 = 10137
Мощность группы 27 = 10337
Мощность группы 28 = 10229
Мощность группы 29 = 9858
Мощность группы 30 = 10099
Мощность группы 31 = 9875
Мощность группы 32 = 9865
Мощность группы 33 = 10141
Мощность группы 34 = 10001
Мощность группы 35 = 9954
Мощность группы 36 = 10076
Мощнос

In [ ]:
df_clean["pace"]

array([14.397    ,  9.059583 ,  3.4483554, ...,  3.940742 , 15.304777 ,
       12.5891905])

In [ ]:
for group in athlete_group:
  subset = df_clean[df_clean["athlete_id"] == group]
  dist_mean = np.nanmean(subset["dist"])
  pace_min = np.nanmin(subset["pace"])
  print(f"Группа {group}:  среднее значение dist{dist_mean:.2f}, минимальный темп {pace_min}")


Группа 0:  среднее значение dist59.98, минимальный темп 1.3435365000000001
Группа 1:  среднее значение dist58.24, минимальный темп 1.3435365000000001
Группа 2:  среднее значение dist58.96, минимальный темп 1.3435365000000001
Группа 3:  среднее значение dist58.86, минимальный темп 1.3435365000000001
Группа 4:  среднее значение dist59.25, минимальный темп 1.3435365000000001
Группа 5:  среднее значение dist58.29, минимальный темп 1.3435365000000001
Группа 6:  среднее значение dist58.89, минимальный темп 1.3435365000000001
Группа 7:  среднее значение dist58.42, минимальный темп 1.3435365000000001
Группа 8:  среднее значение dist58.57, минимальный темп 1.3435365000000001
Группа 9:  среднее значение dist59.06, минимальный темп 1.3435365000000001
Группа 10:  среднее значение dist59.82, минимальный темп 1.3435365000000001
Группа 11:  среднее значение dist59.24, минимальный темп 1.3435365000000001
Группа 12:  среднее значение dist58.07, минимальный темп 1.3435365000000001
Группа 13:  среднее зн

In [ ]:
for group in athlete_group:
  data = df_clean[df_clean["athlete_id"] == group]
  mask = np.isnan(data[col]) | np.isinf(data[col])
  print(f"В группе {group} {mask.sum()} NaN и Inf.", end = " ")
  print()

В группе 0 0 NaN и Inf. 
В группе 1 0 NaN и Inf. 
В группе 2 0 NaN и Inf. 
В группе 3 0 NaN и Inf. 
В группе 4 0 NaN и Inf. 
В группе 5 0 NaN и Inf. 
В группе 6 0 NaN и Inf. 
В группе 7 0 NaN и Inf. 
В группе 8 0 NaN и Inf. 
В группе 9 0 NaN и Inf. 
В группе 10 0 NaN и Inf. 
В группе 11 0 NaN и Inf. 
В группе 12 0 NaN и Inf. 
В группе 13 0 NaN и Inf. 
В группе 14 0 NaN и Inf. 
В группе 15 0 NaN и Inf. 
В группе 16 0 NaN и Inf. 
В группе 17 0 NaN и Inf. 
В группе 18 0 NaN и Inf. 
В группе 19 0 NaN и Inf. 
В группе 20 0 NaN и Inf. 
В группе 21 0 NaN и Inf. 
В группе 22 0 NaN и Inf. 
В группе 23 0 NaN и Inf. 
В группе 24 0 NaN и Inf. 
В группе 25 0 NaN и Inf. 
В группе 26 0 NaN и Inf. 
В группе 27 0 NaN и Inf. 
В группе 28 0 NaN и Inf. 
В группе 29 0 NaN и Inf. 
В группе 30 0 NaN и Inf. 
В группе 31 0 NaN и Inf. 
В группе 32 0 NaN и Inf. 
В группе 33 0 NaN и Inf. 
В группе 34 0 NaN и Inf. 
В группе 35 0 NaN и Inf. 
В группе 36 0 NaN и Inf. 
В группе 37 0 NaN и Inf. 
В группе 38 0 NaN и In

In [ ]:
columns = ["ts", "athlete_id", "dist", "pace", "cal", "zone"]

np.where(df_clean)

for col in columns:
  mask = np.isnan(df_clean[col]) | np.isinf(df_clean[col])
  percent = (mask.sum() / len(df[col])) * 100
  print(f"В столбце {col} {mask.sum()} NaN и Inf.", end = " ")
  if percent > 3:
    print(f"В столбце содержится больше 3%!")
  else:
    print()

В столбце ts 0 NaN и Inf. 
В столбце athlete_id 0 NaN и Inf. 
В столбце dist 1754 NaN и Inf. 
В столбце pace 3227 NaN и Inf. 
В столбце cal 1653 NaN и Inf. 
В столбце zone 0 NaN и Inf. 


Z-score нормализация

In [ ]:
dist_mean = np.nanmean(df["dist"])
dist_std = np.nanstd(df["dist"])

print(f"Original {df_clean["dist"]}")
df_clean["dist"] = (df_clean["dist"] - dist_mean) / dist_std

print(f"New {df_clean["dist"]}")

pace_mean = np.nanmean(df["pace"])
pace_std = np.nanstd(df["pace"])

print(f"Original {df_clean["pace"]}")

df_clean["pace"] = (df_clean["pace"] - pace_mean) / pace_std

print(f"New {df_clean["pace"]}")



Original [75.74629  48.81974  38.624058 ... 65.52158  25.191607 67.86183 ]
New [ 0.34943832 -0.12248715 -0.30118074 ...  0.17023597 -0.53660324
  0.21125213]
Original [12.966491   4.47056    1.3435365 ...  1.6444373  7.3264446 19.257215 ]
New [ 0.47314472  0.07111296 -0.07685938 ... -0.0626206   0.20625487
  0.77082497]


#4. Скользящее окно и разница

In [ ]:
from numpy.lib.recfunctions import append_fields

In [ ]:
k = 30

In [ ]:
cal_dist_ratio = np.where(df_clean["dist"] == 0, np.nan, df_clean["cal"] / df_clean["dist"])

In [ ]:
c = np.concatenate(([0.0], np.cumsum(cal_dist_ratio)))

In [ ]:
ma_ratio = (c[k:] - c[:-k]) / k

In [ ]:
ma_ratio_paddid = np.pad(ma_ratio, (k - 1, 0), constant_values = np.nan)

In [ ]:
pace_diff = np.diff(df_clean["pace"])
pace_diff_padded = np.pad(pace_diff, (1, 0), constant_values = np.nan)

In [ ]:
df_extended = append_fields(
    df_clean,
    names = "pace_diff_padded",
    data = pace_diff_padded,
    dtypes = np.float64,
    usemask = False
)

In [ ]:
df_extended

array([(1600000180, 35,  0.34943832,  0.47314472, 7.4050336, 15,         nan),
       (1600000284, 43, -0.12248715,  0.07111296, 0.       , 94, -0.40203176),
       (1600000357, 30, -0.30118074, -0.07685938, 5.287114 , 77, -0.14797234),
       ...,
       (1749999371, 16,  0.17023597, -0.0626206 ,       nan, 73,         nan),
       (1749999727, 22, -0.53660324,  0.20625487, 0.       , 14,  0.26887547),
       (1749999928, 81,  0.21125213,  0.77082497, 0.       , 53,  0.5645701 )],
      dtype=[('ts', '<i8'), ('athlete_id', '<i4'), ('dist', '<f8'), ('pace', '<f8'), ('cal', '<f8'), ('zone', '<i4'), ('pace_diff_padded', '<f8')])

#5. Создание производных признаков (Feature Engineering)

Расчёт мгновенной скорости

In [ ]:
speed_km = np.where(df_extended["pace"] > 0, 60 / df_extended["pace"], np.nan)

df_extended = append_fields(
    df_extended,
    names = "speed_km",
    data = speed_km,
    dtypes = np.float64,
    usemask = False
)

In [ ]:
df_extended["speed_km"]

array([126.81109439, 843.72807083,          nan, ...,          nan,
       290.90222682,  77.83868224])

In [ ]:
sum(np.isnan(df_extended["speed_km"]))

np.int64(265174)

Замена nan на медиану df_extended["speed_km"]

In [ ]:
df_extended["speed_km"] = np.where(np.isnan(df_extended["speed_km"]), np.nanmedian(df_extended["speed_km"]), df_extended["speed_km"])

In [ ]:
sum(np.isnan(df_extended["speed_km"]))

np.int64(0)

#6. Условная агрегация по группам

In [ ]:
mask = ((df_extended["pace"] > 0) & (df_extended["dist"] > 0) & (~np.isnan(df_extended["dist"])))

df_filter = df_extended[mask]
athlete_id = np.unique(df_filter["athlete_id"])
print(athlete_id)
n_groups = len(athlete_id)
result = np.zeros((n_groups, 4), dtype = np.float64)
result[:, 0] = athlete_id

for i, grid in enumerate(athlete_id):
  group = (df_filter["athlete_id"] == grid)
  values = df_filter["speed_km"][group]

  result[i, 1] = np.nanmean(values)
  result[i, 2] = np.nanmedian(values)
  result[i, 3] = np.nanpercentile(values, 90)








[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71
 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95
 96 97 98 99]


In [ ]:
result

array([[0.00000000e+00, 9.31898374e+02, 1.99095505e+02, 1.12749443e+03],
       [1.00000000e+00, 1.47532923e+03, 2.03927597e+02, 1.09895943e+03],
       [2.00000000e+00, 8.21684277e+02, 2.01742361e+02, 1.28570546e+03],
       [3.00000000e+00, 1.37543707e+03, 2.00587888e+02, 1.27085514e+03],
       [4.00000000e+00, 1.03121361e+03, 2.04002338e+02, 1.16069051e+03],
       [5.00000000e+00, 1.05093350e+03, 2.02504833e+02, 1.12042051e+03],
       [6.00000000e+00, 1.78908123e+03, 1.95461093e+02, 1.20201485e+03],
       [7.00000000e+00, 1.82966756e+03, 1.98251065e+02, 1.20730270e+03],
       [8.00000000e+00, 1.55882788e+03, 1.98856087e+02, 1.11944583e+03],
       [9.00000000e+00, 1.49571256e+03, 1.99287485e+02, 1.13289816e+03],
       [1.00000000e+01, 1.08802817e+03, 1.96466177e+02, 1.14427760e+03],
       [1.10000000e+01, 9.57891354e+02, 2.00823358e+02, 1.14285959e+03],
       [1.20000000e+01, 8.28386488e+02, 1.91869519e+02, 1.03895858e+03],
       [1.30000000e+01, 1.46810111e+03, 1.95987154e

In [ ]:
result

array([[0.00000000e+00, 6.37887046e+02, 1.89273873e+02, 1.25081676e+03],
       [1.00000000e+00, 6.24014052e+02, 1.89978494e+02, 8.88936744e+02],
       [2.00000000e+00, 1.20251812e+03, 2.33690015e+02, 1.67075442e+03],
       [3.00000000e+00, 6.33217290e+02, 1.74909390e+02, 1.34837248e+03],
       [4.00000000e+00, 1.31175495e+03, 1.92693454e+02, 1.33830537e+03],
       [5.00000000e+00, 8.24172615e+02, 2.21382143e+02, 1.26208805e+03],
       [6.00000000e+00, 8.59316600e+02, 1.99792435e+02, 9.77127238e+02],
       [7.00000000e+00, 1.40166664e+03, 2.01761956e+02, 1.03779684e+03],
       [8.00000000e+00, 8.08557149e+02, 2.16798738e+02, 2.16778479e+03],
       [9.00000000e+00, 5.35160709e+02, 1.64738216e+02, 8.93553494e+02],
       [1.00000000e+01, 1.35588912e+03, 1.86686175e+02, 1.40102271e+03],
       [1.10000000e+01, 5.12637485e+02, 2.24755849e+02, 9.65136778e+02],
       [1.20000000e+01, 5.31646616e+02, 2.13616293e+02, 1.09973892e+03],
       [1.30000000e+01, 6.85016005e+02, 2.00687009e

In [ ]:
df_extended

array([(1600000063, 38,  0.00538159, 0.53611626, 10.340377  , 95,         nan,  111.91602345),
       (1600000210, 58,  0.41694621, 0.28468983,  0.        , 91, -0.25142643,  210.75568686),
       (1600000430, 91,  0.61765425, 0.02036516,  0.        , 22, -0.26432466, 2946.20743725),
       ...,
       (1608334306, 52, -0.92599089, 0.04355972,  0.        , 45, -0.31218698, 1377.41935976),
       (1608334318, 71, -0.71178379, 0.57887836,  0.34645525, 17,  0.53531864,  103.64871893),
       (1608334735, 77, -0.56137751, 0.45095689,  8.4685    , 11, -0.12792147,  133.05041325)],
      dtype=[('ts', '<i8'), ('athlete_id', '<i4'), ('dist', '<f8'), ('pace', '<f8'), ('cal', '<f8'), ('zone', '<i4'), ('pace_diff_padded', '<f8'), ('speed_km', '<f8')])

#7.Лаговые признаки и анализ временных сдвигов

In [ ]:
pace_val = np.roll(df_extended["pace"], 1)

pace_val[0] = np.nan

In [ ]:
df_extended = append_fields(df_extended, "предыдущее значение", pace_val, usemask = False)

Вычислям разницу между текущим и лаговым значением.

In [ ]:
diff = df_extended["pace"] - df_extended["предыдущее значение"]


Находим долю записей, где значение выросло/упало по сравнению с предыдущим замером

In [ ]:
n = len(diff)

print(f"Выросло {np.sum(diff > 0) / n}")
print(f"Упало {np.sum(diff < 0) / n}")

Выросло 0.4353367727800154
Упало 0.43434803994440746


 распределение знаков разницы

In [ ]:
valid_diff = diff[~np.isnan(diff)]
signs = np.sign(valid_diff)

uniq_value, counts = np.unique(signs, return_counts = True)
sign_dict = {
    -1.0: "Упало",
    0: "Осталось неизменным",
    1: "Выросло"
}

for v, c in zip(uniq_value, counts):
  print(f"{sign_dict[v]}: {c}")

Упало: 435344
Осталось неизменным: 17535
Выросло: 436335


#8.Групповая робастная замена выбросов

In [ ]:
columns = ["ts", "athlete_id", "dist", "pace", "cal", "zone"]
replaced_counts = {}
total_replaced = 0
total_cells = 0

for col in columns:
  col_data = df_extended[col]
  col_sort = np.sort(col_data)
  median_col = np.nanmedian(col_sort)
  Q1 = np.nanpercentile(col_sort, 25)
  Q3 = np.nanpercentile(col_sort, 75)
  IQR = Q3 - Q1
  lower_limit = Q1 - 1.5 * IQR
  upper_limit = Q3 + 1.5 * IQR
  print(f"Столбец {col}")
  print(f"Медиана {median_col}")
  print(f"Q1 = {Q1}")
  print(f"Q3 = {Q3}")
  print(f"Нижняя граница {lower_limit}")
  print(f"Верхняя граница {upper_limit}")
  print()
  is_outlier = (col_data < lower_limit) | (col_data > upper_limit)
  n_replaced = np.sum(is_outlier)
  replaced_counts[col] = n_replaced
  total_replaced += n_replaced
  total_cells += len(col_data)

  df_extended[col] = np.where(is_outlier, median_col, col_data)

for col, cnt in replaced_counts.items():
  print(f"{col} : {cnt}")

result = total_replaced / total_cells
print(f"Общая доля изменённых значений: {result}")



Столбец ts
Медиана 1675020272.0
Q1 = 1637468470.0
Q3 = 1712492083.0
Нижняя граница 1524933050.5
Верхняя граница 1825027502.5

Столбец athlete_id
Медиана 50.0
Q1 = 24.0
Q3 = 75.0
Нижняя граница -52.5
Верхняя граница 151.5

Столбец dist
Медиана -0.08376842267088416
Q1 = -0.3853065215198848
Q3 = 0.2277825431698513
Нижняя граница -1.3049401185544889
Верхняя граница 1.1474161402044554

Столбец pace
Медиана 0.2025272457545355
Q1 = 0.020469417027377347
Q3 = 0.4618780878170908
Нижняя граница -0.6416435891571928
Верхняя граница 1.1239910940016609

Столбец cal
Медиана 0.018865606
Q1 = 0.0
Q3 = 7.2509866
Нижняя граница -10.8764799
Верхняя граница 18.1274665

Столбец zone
Медиана 49.0
Q1 = 24.0
Q3 = 74.0
Нижняя граница -51.0
Верхняя граница 149.0

ts : 0
athlete_id : 0
dist : 31992
pace : 0
cal : 60980
zone : 0
Общая доля изменённых значений: 0.015459883819734682


#9.Проверка согласованности и логической целостности

In [ ]:
rule_dist = df_extended["dist"] > 0
rule_zone = (df_extended['zone'] >= 1) & (df_extended['zone'] <= 5)

valid_mask = rule_dist & rule_zone

mask = ~valid_mask

n_violations = np.sum(mask)
prop_violations = np.sum(mask) / len(df_extended)

print(f"Абсолютное кол-во нарушений {n_violations}")
print(f"Доля нарушений {prop_violations}")

valid_dist =  df_extended["dist"][rule_dist]
ref_dist = np.nanmedian(valid_dist)

vals, cnt = np.unique(df_extended["zone"], return_counts = True)

ref_zone = np.uint8(vals[np.argmax(cnt)])

df_extended["dist"] = np.where(
    ~rule_dist,
    ref_dist,
    df_extended["dist"]
)

df_extended["zone"] = np.where(
    ~rule_zone,
    ref_zone,
    df_extended["zone"]
)




Абсолютное кол-во нарушений 983090
Доля нарушений 0.9808409317435122


In [ ]:
df_extended

array([(1600000180, 35, 0.34943832,  0.47314472, 7.4050336, 47,         nan, 126.81109439,         nan),
       (1600000284, 43, 0.25911024,  0.07111296, 0.       , 47, -0.40203176, 843.72807083,  0.47314472),
       (1600000357, 30, 0.25911024, -0.07685938, 5.287114 , 47, -0.14797234, 200.35808054,  0.07111296),
       ...,
       (1749999371, 16, 0.17023597, -0.0626206 ,       nan, 47,         nan, 200.35808054,         nan),
       (1749999727, 22, 0.25911024,  0.20625487, 0.       , 47,  0.26887547, 290.90222682, -0.0626206 ),
       (1749999928, 81, 0.21125213,  0.77082497, 0.       , 47,  0.5645701 ,  77.83868224,  0.20625487)],
      dtype=[('ts', '<i8'), ('athlete_id', '<i4'), ('dist', '<f8'), ('pace', '<f8'), ('cal', '<f8'), ('zone', '<i4'), ('pace_diff_padded', '<f8'), ('speed_km', '<f8'), ('предыдущее значение', '<f8')])

#10.Частотный анализ и сжатие редких категорий

In [ ]:
group, count = np.unique(df_extended["zone"], return_counts = True)
groups = dict(zip(group, count))
other = []
for gr, val in groups.items():
  prop = val / len(df_extended)
  if prop < 0.01:
    other.append(gr)

  print(f"Группа:{gr} Доля {prop}")


df_extended["zone"] = np.where(np.isin(df_extended["zone"], other), 0, df_extended["zone"])

np.unique(df_extended["zone"])

Группа:1 Доля 0.010038980617444201
Группа:2 Доля 0.010124783870584748
Группа:3 Доля 0.00989032149281697
Группа:4 Доля 0.01009385479096432
Группа:5 Доля 0.009961159062270214
Группа:47 Доля 0.9498909001659196


array([ 0,  1,  2,  4, 47], dtype=int32)